## Structured Output

Models can be request to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain Supports Multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature with field validation, description and nested structure.


In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")  # reasoning model
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f881940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f882660>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="This is the title of the movie")
    year:int=Field(description="This is the year of release")
    director:str=Field(description="This is the director of the movie")
    ratings:float=Field(description="This is the rating of movie out of 10")

In [12]:
model_with_structure = model.with_structured_output(Movie,include_raw=True)
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f881940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f882660>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'de

In [9]:
response = model.invoke("provide me with the details of movie Hera Pheri")
response.usage_metadata

{'input_tokens': 19, 'output_tokens': 1558, 'total_tokens': 1577}

In [13]:
response = model_with_structure.invoke("Provide me with the details of movie Hera Pheri")
print(response["raw"].usage_metadata)  

{'input_tokens': 236, 'output_tokens': 251, 'total_tokens': 487, 'output_token_details': {'reasoning': 198}}


### Message Output alongside parsed structure

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(...,description="This is the title of the movie")
    year:int=Field(description="This is the year of release")
    director:str=Field(description="This is the director of the movie")
    ratings:float=Field(description="This is the rating of movie out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Provide me with the details of movie Hera Pheri")
# print(response)
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Hera Pheri". Let me check if I have that information. The available function is called "Movie" and it requires the parameters title, year, director, and ratings.\n\nFirst, I need to recall the details of "Hera Pheri". I remember it\'s a popular Indian comedy film. The director is Priyadarshan. The release year was 2000. The main actors are Akshay Kumar, John Abraham, and Emraan Hashmi. The ratings on IMDb are around 6.9 out of 10. \n\nWait, let me make sure about the director. Yes, Priyadarshan directed it. The year is definitely 2000. The ratings might vary slightly depending on the source, but IMDb lists it as 6.9. I should use that. \n\nSo, putting it all together, the function call should include the title "Hera Pheri", year 2000, director "Priyadarshan", and ratings 6.9. I need to structure this into the required JSON format as specified by the tool. Let me 

### Nested Output

In [20]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget for creating the movie in Indian Rupees INR")



model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide me with the details of movie Hera Pheri")
response

MovieDetails(title='Hera Pheri', year=2000, cast=[Actor(name='Akshay Kumar', role='Shyam'), Actor(name='Salman Khan', role='Raju'), Actor(name='Zayed Khan', role='Bunty'), Actor(name='Deepika Padukone', role='Pinki')], genre=['Comedy', 'Action'], budget=15000000.0)